# Early updates analysis

How the first updates after the critic warm-up move the actor (and, in later cells, the critics) of one DDPG agent,
from a set of its checkpoints -- by default `ckpt_critic_warmup.ckpt` and every `ckpt_rollout_<k>_<step>.ckpt`
(`ddpg.init_rollout_checkpoint_saves`) -- evaluated on the replay buffer saved with the LAST of them, the demos
included.

The last checkpoint's buffer must be on disk: `memory_<name>.pt` beside `ckpt_<name>.ckpt`
(`ddpg.save_memory_with_last_rollout_checkpoint` for a rollout checkpoint, `save_memory_with_critic_warmup` for the
warm-up's, `save_memory_with_checkpoints` for a numbered one). No sim: the networks are rebuilt from the run's
`resolved_config.yaml` and the checkpoints' weights.

Cells: 1 imports, 2 variables, 3 loading, 4 preprocessing, then one plot or table per cell (5: the actors' action
histograms over the checkpoints).

In [ ]:
import copy
import re
import sys
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from omegaconf import OmegaConf

# the repo root, so the project's config and residual code import
REPO = Path.cwd().resolve()
while not (REPO / "my_project").is_dir():
    if REPO.parent == REPO:
        raise RuntimeError("run this notebook from inside the hybrid_demos repo")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from robonuke_rl_core.config import SECTIONS  # noqa: E402  (the package's section dataclasses)
from robonuke_rl_core.learners.ddpg import DDPG  # noqa: E402
from robonuke_rl_core.learners.cfg import sac_critic_outputs  # noqa: E402
from robonuke_rl_core.learners.sac import critic_action_space  # noqa: E402
from robonuke_rl_core.models.cfg import model_cfg_class  # noqa: E402
from robonuke_rl_core.models.factory import build_models  # noqa: E402
from my_project.cfg import ResidualCfg  # noqa: E402
from my_project.pose_residual import pose_from  # noqa: E402
from my_project.residual import PoseSideInput, base_alone_r  # noqa: E402

In [ ]:
# the agent: one agent's run directory (runs/<wandb project>/<group>/<group>_a<i>); its group directory holds the
# run's resolved_config.yaml
RUN_DIR = REPO / "runs" / "one_leg_scaled_ablations" / "smoke_rollouts" / "smoke_rollouts_a0"

# a routed run (ddpg.route_labels): the phase whose residual to analyse (its checkpoints are in RUN_DIR/<phase>/);
# None for an unrouted run
ROUTE = None

# the checkpoints, by file name in the agent's checkpoint folder; None: ckpt_critic_warmup.ckpt and every
# ckpt_rollout_<k>_<step>.ckpt. They are ordered by their step; the buffer is the last one's
CHECKPOINTS = None

# where the networks run
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# observations per forward pass (each runs every checkpoint's actor on them at once)
BATCH_ROWS = 4096

# the figures: size in inches per subplot row, resolution, font (the first family installed is used)
FIG_WIDTH = 11
ROW_HEIGHT = 1.8
DPI = 150
FONT_FAMILY = ["Times New Roman", "Liberation Serif", "DejaVu Serif"]
FONT_SIZE = 11

In [ ]:
ckpt_dir = (RUN_DIR / ROUTE if ROUTE else RUN_DIR) / "checkpoints"
if not ckpt_dir.is_dir():
    raise FileNotFoundError(f"no checkpoint folder {ckpt_dir}")

# the checkpoint set, ordered by step
if CHECKPOINTS is None:
    rollout = re.compile(r"ckpt_rollout_(\d+)_(\d+)\.ckpt")
    names = (["ckpt_critic_warmup.ckpt"] if (ckpt_dir / "ckpt_critic_warmup.ckpt").exists() else [])
    names += sorted((p.name for p in ckpt_dir.glob("ckpt_rollout_*.ckpt") if rollout.fullmatch(p.name)),
                    key=lambda n: int(rollout.fullmatch(n).group(1)))
else:
    names = list(CHECKPOINTS)
if not names:
    raise FileNotFoundError(f"no checkpoints to analyse in {ckpt_dir} (CHECKPOINTS={CHECKPOINTS})")
missing = [n for n in names if not (ckpt_dir / n).is_file()]
if missing:
    raise FileNotFoundError(f"not in {ckpt_dir}: {missing}")
ckpts = sorted(((n, torch.load(ckpt_dir / n, map_location="cpu", weights_only=False)) for n in names),
               key=lambda item: item[1]["step"])
steps = np.array([c["step"] for _, c in ckpts])
if len(set(steps.tolist())) != len(steps):
    raise ValueError(f"two checkpoints share a step: {dict(zip([n for n, _ in ckpts], steps.tolist()))}")

def short_label(name):
    """warm-up, r<k>, or the step of a numbered checkpoint."""
    if name == "ckpt_critic_warmup.ckpt":
        return "warm-up"
    m = re.fullmatch(r"ckpt_rollout_(\d+)_\d+\.ckpt", name)
    return f"r{m.group(1)}" if m else name.removeprefix("ckpt_").removesuffix(".ckpt")

labels = [short_label(n) for n, _ in ckpts]

# the buffer: the one saved with the last checkpoint, memory_<name>.pt beside ckpt_<name>.ckpt
last_name, last_ckpt = ckpts[-1]
buffer_path = ckpt_dir / ("memory_" + last_name.removeprefix("ckpt_").removesuffix(".ckpt") + ".pt")
if not buffer_path.is_file():
    raise FileNotFoundError(
        f"{last_name} has no saved buffer ({buffer_path.name} is not in {ckpt_dir}): train with "
        "ddpg.save_memory_with_last_rollout_checkpoint (a rollout checkpoint), ddpg.save_memory_with_critic_warmup "
        "(the warm-up's) or ddpg.save_memory_with_checkpoints (a numbered one)")
buffer = torch.load(buffer_path, map_location="cpu", weights_only=False)
if buffer["meta"]["step"] != last_ckpt["step"]:
    raise ValueError(f"{buffer_path.name} is from step {buffer['meta']['step']}, {last_name} from {last_ckpt['step']}")
if "demos" not in buffer:
    raise KeyError(f"{buffer_path.name} was saved before buffers carried the demos: retrain with the current package")

# the run's config: the sections the networks need, without the task (no sim)
group_dir = RUN_DIR.parent
raw = OmegaConf.load(group_dir / "resolved_config.yaml")

def section(name, cls):
    return OmegaConf.to_object(OmegaConf.merge(OmegaConf.structured(cls), raw[name]))

if raw.trainer.learner != "ddpg":
    raise ValueError(f"this notebook reads DDPG runs; {group_dir.name} is trainer.learner {raw.trainer.learner}")
trainer_cfg = section("trainer", SECTIONS["trainer"])
model_cfg = section("model", model_cfg_class(raw.model.architecture))
ddpg_cfg = section("ddpg", SECTIONS["ddpg"])
residual_cfg = section("residual", ResidualCfg)
if residual_cfg.action_space != "pose":
    raise NotImplementedError("the critics' action transform is rebuilt here for residual.action_space: pose only "
                              f"(this run: {residual_cfg.action_space})")
if ddpg_cfg.route_labels and ROUTE not in ddpg_cfg.route_labels:
    raise ValueError(f"this run is routed ({list(ddpg_cfg.route_labels)}): set ROUTE to one of them")
if not ddpg_cfg.route_labels and ROUTE is not None:
    raise ValueError(f"this run is not routed: set ROUTE = None (got {ROUTE!r})")

# the learner's own section as training built it (train.py: one learner per route)
learner_cfg = copy.deepcopy(ddpg_cfg)
if ROUTE:
    learner_cfg.route_label, learner_cfg.route_labels = ROUTE, []
learner_cfg.demo_transitions = None  # the demos come from the buffer, as the learner held them
learner_cfg.cuda_graph = False

# the critics' action: the resulting target relative to the EE and the gripper, from the side input (the furniture
# base action's gripper is its dim 9)
action_range = tuple(float(v) for v in ddpg_cfg.action_range)
pose = pose_from(residual_cfg, action_range, gripper=9)
transform = PoseSideInput(pose)

# the spaces, from the buffer's widths
rows = buffer["rows"]
box = lambda width: gym.spaces.Box(-np.inf, np.inf, (int(width),), np.float32)  # noqa: E731
observation_space = box(rows["observations"].shape[-1])
state_space = box(rows["states"].shape[-1]) if "states" in rows else None
action_space = box(rows["actions"].shape[-1])

# one learner, one slot per checkpoint (slot i = ckpts[i])
N = len(ckpts)
models = build_models("ddpg", model_cfg, observation_space, state_space, action_space, N, DEVICE, None,
                      num_critics=int(learner_cfg.num_critics), critic_outputs=sac_critic_outputs(learner_cfg),
                      critic_action_space=critic_action_space(transform, action_space))
learner = DDPG(models=models, memory=None, observation_space=observation_space, state_space=state_space,
               action_space=action_space, device=DEVICE, cfg=learner_cfg, trainer_cfg=trainer_cfg, num_agents=N,
               num_envs=N, model_cfg=model_cfg, controller_cfg=None,
               critic_action_space=critic_action_space(transform, action_space))
learner.critic_action_transform = transform
learner.init_side_inputs()
for slot, (name, _) in enumerate(ckpts):
    learner.load_agent(ckpt_dir / name, slot=slot, with_optimizer=False)
learner.enable_training_mode(False)

print(f"{N} checkpoints from {ckpt_dir}:")
for (name, c), label in zip(ckpts, labels):
    print(f"  {label:>8}  step {c['step']:>7}  {name}")
meta = buffer["meta"]
print(f"buffer {buffer_path.name}: {meta['rows']} online rows (capacity {meta['capacity']}, wrapped {meta['wrapped']}), "
      f"{meta.get('demo_rows_separate', 0)} demo rows beside them, {meta['demo_rows_in_memory']} demo rows among them")

In [ ]:
# every observation the critics trained on at the last checkpoint: the buffer's online rows and the demo rows (with
# demo_storage: sample beside them, with memory / memory_critic its first demo_rows_in_memory rows)
fields = ["observations"] + (["states"] if state_space is not None else []) + sorted(learner._side_inputs) + ["actions"]
in_memory = int(meta["demo_rows_in_memory"])
parts = {"online": {f: rows[f][in_memory:] for f in fields}}
if in_memory:
    parts["demo"] = {f: rows[f][:in_memory] for f in fields}
elif buffer["demos"] is not None:
    parts["demo"] = {f: buffer["demos"][f] for f in fields}
elif ddpg_cfg.demo_transitions:
    raise ValueError(f"the run loads demos ({ddpg_cfg.demo_transitions}) but {buffer_path.name} holds none")
else:
    print("note: this run has no demos; only the online rows are analysed")

data = {f: torch.cat([p[f] for p in parts.values()]).float() for f in fields}
source = np.concatenate([np.full(len(p["observations"]), s) for s, p in parts.items()])
for s in parts:
    print(f"{s}: {int((source == s).sum())} rows")

# the r dims' names: the position step (3) then the rotation's numbers (residual.rotation_param), in the action frame
rot = {"rotvec": ["rx", "ry", "rz"], "euler": ["roll", "pitch", "yaw"], "rot6d": [f"r6d_{i}" for i in range(6)]}
dim_names = ["x", "y", "z"] + rot[residual_cfg.rotation_param]
if len(dim_names) != action_space.shape[0]:
    raise ValueError(f"r has {action_space.shape[0]} dims, the pose names {len(dim_names)}")

plt.rcParams.update({"font.family": "serif", "font.serif": FONT_FAMILY, "font.size": FONT_SIZE})

@torch.no_grad()
def actor_actions(observations):
    """Every checkpoint's eval action (DDPG's clamped mean, no noise) on ``observations`` ``(R, D)``: ``(N, R, A)``."""
    out = []
    for start in range(0, len(observations), BATCH_ROWS):
        chunk = observations[start:start + BATCH_ROWS].to(DEVICE)
        acted = learner.eval_actions(chunk.repeat(N, 1))  # agent-major: slot i's rows are block i
        out.append(acted.view(N, len(chunk), -1).cpu())
    return torch.cat(out, dim=1)

In [ ]:
# Each checkpoint's actor on every buffer observation: per r dim, the histogram of its actions, one column per
# checkpoint (wandb's histogram panel: x the training step, y the action, the colour each column's share of the
# rows in that bin). The dashed line is the r that runs the base alone (1 for multiply, 0 for add).

# which rows: "online", "demo" or both
SOURCES = ("online", "demo")
# the bins: a count, or explicit edges (then RANGE is ignored)
BINS = 60
# the y range per dim: None for ddpg.action_range on every dim, one (lo, hi) for all dims, or one (lo, hi) per dim
RANGE = None
# the x axis: "step" (each column centred on its checkpoint's env step) or "index" (evenly spaced, labelled)
X_AXIS = "index"
# the colour: each column's fraction of rows per bin, on a log scale or not
LOG_COLOUR = False
COLOUR_MAP = "viridis"

keep = np.isin(source, SOURCES)
if not keep.any():
    raise ValueError(f"no rows from {SOURCES}; the buffer has {sorted(set(source.tolist()))}")
acts = actor_actions(data["observations"][torch.from_numpy(keep)]).numpy()  # (N, R, A)
A = acts.shape[-1]

if RANGE is None:
    ranges = [action_range] * A
elif np.ndim(RANGE) == 1:
    ranges = [tuple(RANGE)] * A
else:
    ranges = [tuple(r) for r in RANGE]
if len(ranges) != A:
    raise ValueError(f"RANGE gives {len(ranges)} ranges for {A} dims")

if X_AXIS == "step":
    centres = steps.astype(float)
    gaps = np.diff(centres) if N > 1 else np.array([1.0])
    x_edges = np.concatenate(([centres[0] - gaps[0] / 2], centres[:-1] + gaps / 2, [centres[-1] + gaps[-1] / 2]))
elif X_AXIS == "index":
    centres = np.arange(N, dtype=float)
    x_edges = np.arange(N + 1) - 0.5
else:
    raise ValueError(f"X_AXIS must be 'step' or 'index', got {X_AXIS!r}")

fig, axes = plt.subplots(A, 1, figsize=(FIG_WIDTH, ROW_HEIGHT * A), dpi=DPI, sharex=True, squeeze=False)
neutral = base_alone_r(residual_cfg)
for d, ax in enumerate(axes[:, 0]):
    edges = np.asarray(BINS, dtype=float) if np.ndim(BINS) else np.linspace(*ranges[d], int(BINS) + 1)
    counts = np.stack([np.histogram(acts[i, :, d], bins=edges)[0] for i in range(N)])  # (N, bins)
    outside = 1.0 - counts.sum(axis=1) / acts.shape[1]
    if (outside > 0).any():
        print(f"{dim_names[d]}: rows outside [{edges[0]:g}, {edges[-1]:g}] per checkpoint: "
              + ", ".join(f"{l} {o:.1%}" for l, o in zip(labels, outside) if o > 0))
    share = counts / acts.shape[1]
    norm = None
    if LOG_COLOUR:
        from matplotlib.colors import LogNorm
        share = np.where(share > 0, share, np.nan)
        norm = LogNorm(vmin=np.nanmin(share), vmax=np.nanmax(share))
    mesh = ax.pcolormesh(x_edges, edges, share.T, cmap=COLOUR_MAP, norm=norm, shading="flat")
    if edges[0] <= neutral <= edges[-1]:
        ax.axhline(neutral, color="white", lw=0.6, ls="--", alpha=0.6)
    ax.set_ylabel(dim_names[d])
    fig.colorbar(mesh, ax=ax, pad=0.01, label="share")
axes[-1, 0].set_xticks(centres)
axes[-1, 0].set_xticklabels([f"{l}\n{s}" for l, s in zip(labels, steps)] if X_AXIS == "index" else steps.tolist(),
                            fontsize=FONT_SIZE - 2)
axes[-1, 0].set_xlabel("checkpoint (env step)" if X_AXIS == "index" else "env step")
fig.suptitle(f"{RUN_DIR.name}{' / ' + ROUTE if ROUTE else ''}: actions on {keep.sum()} buffer rows ({', '.join(SOURCES)})")
fig.tight_layout()
plt.show()